# 🚀 DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic
اجرای مدل بدون سانسور روی پردازنده گرافیکی گوگل کولب با رابط وب چت (Gradio)

In [ ]:
# ۱. نصب کتابخانه‌ها
!pip install -q transformers torch accelerate gradio

In [ ]:
# ۲. بارگذاری و اجرای مدل با رابط کاربری وب
import torch
from threading import Thread
import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer

MODEL_ID = "xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic"

print(f"⏳ در حال بارگذاری توکنایزر {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

print(f"⏳ در حال بارگذاری مدل روی پردازنده گرافیکی (GPU)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)
print("✅ مدل با موفقیت روی حافظه گرافیکی بارگذاری شد!")

def predict(message, history):
    messages = []
    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})
    messages.append({"role": "user", "content": message})

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    streamer = TextIteratorStreamer(tokenizer, timeout=60.0, skip_prompt=True, skip_special_tokens=True)

    generation_kwargs = dict(
        inputs,
        streamer=streamer,
        max_new_tokens=1024,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    partial_message = ""
    for new_token in streamer:
        partial_message += new_token
        yield partial_message

demo = gr.ChatInterface(
    fn=predict,
    title="DeepSeek-V4-Pro-Qwen3.5-9B (Uncensored)",
    description="اجرا شده روی پردازنده گرافیکی گوگل کولب",
)

demo.queue().launch(share=True)